# Les files — Exercice 4 : recherche et suppression des doublons (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** recherche dans une file, file auxiliaire, réutiliser ses propres fonctions.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# MISE EN PLACE

> Dans la suite, on utilisera exclusivement uniquement des objets de la classe File définie dans l'exercice 1.
> (Interdit d'utiliser le type list de python.)

On importe la classe `File` de l'exercice 1 : le fichier `file.py` doit être **dans le même dossier** que ce notebook.

**Aucune liste python, aucune boucle `for`** : on ne manipule les files qu'avec `enfile`, `defile`, `est_vide` (et `taille`). Pour les vérifications, on fabrique les files par des `enfile` successifs, et on compare leur **affichage** au texte attendu.

Une file s'affiche sous la forme `File(['a', 'b', 'c'])` : la **tête** est le premier élément (à gauche, c'est lui qu'on défile), la **queue** est le dernier (à droite, c'est là qu'on enfile).

In [1]:
from file import File

f = File()
f.enfile("a")
f.enfile("b")
f.enfile("c")
print(f)

File(['a', 'b', 'c'])


---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Écrire une fonction `est_dans_la_file(f, val)` qui renvoie `True` si la valeur `val` apparaît au moins une fois dans la file `f`, et renvoie `False` sinon. Après l'appel, la file `f` doit être identique à ce qu'elle était initialement.

#### ✅ RÉSOLUTION

In [2]:
def est_dans_la_file(f, val):
    aux = File()
    trouve = False
    while not f.est_vide():
        el = f.defile()
        if el == val:
            trouve = True
        aux.enfile(el)
    while not aux.est_vide():
        f.enfile(aux.defile())
    return trouve

*Vérification (non demandée) :*

In [3]:
f = File()
f.enfile(5)
f.enfile(8)
f.enfile(2)
f.enfile(8)
print(est_dans_la_file(f, 2), est_dans_la_file(f, 7))
print("La file après les appels :", f)

assert est_dans_la_file(f, 2)
assert not est_dans_la_file(f, 7)
assert str(f) == "File([5, 8, 2, 8])"   # file inchangée
assert not est_dans_la_file(File(), 1)

True False
La file après les appels : File([5, 8, 2, 8])


#### 💡 EXPLICATION PÉDAGOGIQUE

- **La technique clé de tout le TD : la file auxiliaire.** Avec une file, on ne voit que la **tête**. Pour regarder tous les éléments, on **vide** la file dans une file auxiliaire `aux`, en traitant chaque élément au passage, puis on **reverse** `aux` dans la file de départ. Comme une file est FIFO, les éléments ressortent de `aux` **dans le même ordre** : la file est **identique** à ce qu'elle était au départ.
  ```python
  aux = File()
  while not f.est_vide():
      el = f.defile()
      ...                    # traiter el
      aux.enfile(el)
  while not aux.est_vide():
      f.enfile(aux.defile())
  ```
- **Pourquoi la première boucle s'arrête ?** Parce qu'on enfile dans `aux`, et **pas** dans `f` : `f` perd un élément à chaque tour et finit vide. ⚠️ Si on ré-enfilait `el` dans `f`, elle ne se viderait jamais et la boucle `while not f.est_vide()` ne s'arrêterait **jamais**.
- **Pourquoi pas `for i in range(f.taille())` ?** Ce serait parcourir la file « case par case », comme un tableau. Une file ne s'utilise pas ainsi : on défile **tant qu'elle n'est pas vide**, comme une vraie file d'attente.
- **Coût :** chaque élément est défilé et enfilé deux fois (aller dans `aux`, retour dans `f`) : de l'ordre de `2n` opérations, donc un coût **linéaire** (proportionnel à `n`). En mémoire, presque rien de plus : les éléments ne sont pas **copiés**, ils **passent** d'une file à l'autre ; à tout moment, `f` et `aux` contiennent ensemble les `n` éléments.
- ⚠️ **Erreur fréquente :** oublier la deuxième boucle. Le résultat est juste, mais `f` est **vide** à la fin, ce que l'énoncé interdit.

- Le booléen `trouve` passe à `True` dès qu'on rencontre `val`, mais **on vide quand même toute la file** dans `aux`, puis on la reverse.
- ⚠️ **Le piège de cet exercice :** avec une liste, on écrirait `return True` dès qu'on trouve la valeur. Ici, la fonction s'arrêterait au milieu du travail : les éléments déjà passés dans `aux` seraient **perdus**, et `f` ne contiendrait plus que la fin de la file. Démonstration :

In [4]:
def est_dans_la_file_FAUX(f, val):
    aux = File()
    while not f.est_vide():
        el = f.defile()
        aux.enfile(el)
        if el == val:
            return True         # on sort au milieu du travail !
    while not aux.est_vide():
        f.enfile(aux.defile())
    return False


f = File()
f.enfile(5)
f.enfile(8)
f.enfile(2)
f.enfile(8)
est_dans_la_file_FAUX(f, 8)
print(f, " <- 5 et 8 ont disparu : ils sont restés dans aux")

File([2, 8])  <- 5 et 8 ont disparu : ils sont restés dans aux


---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Écrire une fonction `supprime_doublons` qui prend en argument une file `f` et la modifie en place de telle sorte qu'elle ne contienne plus de doublons.

#### ✅ RÉSOLUTION

On garde la **première apparition** de chaque valeur, dans l'ordre. On vide `f` dans une file auxiliaire `vues`, en n'y mettant que les valeurs **pas encore vues** ; puis on reverse `vues` dans `f`.

In [5]:
def supprime_doublons(f):
    vues = File()
    while not f.est_vide():
        el = f.defile()
        if not est_dans_la_file(vues, el):
            vues.enfile(el)
    while not vues.est_vide():
        f.enfile(vues.defile())

*Vérification (non demandée) :*

In [6]:
f = File()
f.enfile(3)
f.enfile(1)
f.enfile(3)
f.enfile(2)
f.enfile(1)
f.enfile(3)
supprime_doublons(f)
print(f)
assert str(f) == "File([3, 1, 2])"

g = File()
g.enfile("a")
g.enfile("a")
g.enfile("a")
supprime_doublons(g)
assert str(g) == "File(['a'])"
h = File()
supprime_doublons(h)
assert h.est_vide()

File([3, 1, 2])


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Pour reconnaître un doublon, il faut se souvenir** des valeurs déjà vues. Les listes sont interdites… mais pas les **files** : la file auxiliaire `vues` sert **à la fois** de mémoire et de résultat.
- On vide `f` : un élément **déjà vu** n'est mis nulle part (il disparaît) ; sinon, on l'enfile dans `vues`. On **réutilise** `est_dans_la_file` de la question 1 (qui laisse `vues` intacte).

| Élément défilé | Déjà vu ? | `vues` |
|:---|:---|:---|
| 3 | non | `3` |
| 1 | non | `3 1` |
| 3 | **oui** → supprimé | `3 1` |
| 2 | non | `3 1 2` |
| 1 | **oui** → supprimé | `3 1 2` |
| 3 | **oui** → supprimé | `3 1 2` |

  À la fin, on reverse `vues` dans `f`, qui contient alors `3 1 2` ✔.
- C'est bien **en place** : c'est le **même objet** `f` qui contient le résultat à la fin ; `vues` n'est qu'un outil de travail.
- ⚠️ **Erreur fréquente :** écrire `return vues` au lieu de reverser `vues` dans `f` : la file reçue serait vide après l'appel.
- **Coût :** pour chacun des `n` éléments, `est_dans_la_file` parcourt `vues` (jusqu'à `n` éléments) : de l'ordre de `n²` opérations (coût **quadratique**).